# AutoData — 100k real-data validation (T4)

This is the **post-audit validation notebook**. It does not rerun every exploratory ablation.

Goals:
1. use real transaction data (`fraudTrain.csv` recommended);
2. verify E2 history/sequence features are point-in-time safe, including simultaneous timestamps and merchant history;
3. keep training fraud-enriched for learnability, but use **natural-prevalence validation and test samples**;
4. compare only the configurations justified by the 30k research run;
5. export a compact, reproducible validation table.

Primary metric: **PR-AUC**. Do not interpret accuracy alone for fraud.


In [ ]:
# 0) Confirm a Colab GPU runtime
import os, sys, subprocess, textwrap, json, shutil, zipfile
from pathlib import Path

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
except Exception as e:
    print("Torch check failed before setup:", e)

!nvidia-smi || true

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Colab: Runtime > Change runtime type > choose T4/L4/A100, then rerun.")


In [ ]:
# 1) Locate or upload the project
# Supported paths:
#   A) /content/transaction-data-intelligence already exists
#   B) /content/v2.zip already exists
#   C) upload the project ZIP when prompted

from pathlib import Path
import os, zipfile

CONTENT = Path("/content")
PROJECT_ROOT = CONTENT / "transaction-data-intelligence"
ZIP_CANDIDATES = [CONTENT / "v2.zip", CONTENT / "transaction-data-intelligence.zip"]

if not (PROJECT_ROOT / "src").exists():
    zpath = next((p for p in ZIP_CANDIDATES if p.exists()), None)
    if zpath is None:
        from google.colab import files
        print("Upload the latest project ZIP (for example v2_updated.zip)")
        uploaded = files.upload()
        names = list(uploaded)
        if not names:
            raise RuntimeError("No ZIP uploaded")
        zpath = CONTENT / names[0]

    print("Extracting", zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(CONTENT)

# Some ZIPs may contain one wrapper directory. Find the actual project root deterministically.
if not (PROJECT_ROOT / "src").exists():
    matches = [p for p in CONTENT.rglob("config.yaml") if (p.parent / "src").exists()]
    if len(matches) != 1:
        raise RuntimeError(f"Could not uniquely locate project root. Found: {matches}")
    PROJECT_ROOT = matches[0].parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())


In [ ]:
# 2) Install the backend dependencies used by this repository
# Core benchmark does NOT require transformers/peft/torchao.
# Keeping the core environment small avoids optional-package version conflicts.

!pip install -q -r requirements.txt

# Restart is usually not needed. Re-import torch and confirm CUDA.
import torch
assert torch.cuda.is_available(), "CUDA disappeared after install"
DEVICE = "cuda"
print("GPU ready:", torch.cuda.get_device_name(0))


## Data

Upload the original, minimally processed dataset. For development, prefer `fraudTrain.csv` rather than `fraudTest.csv` so the latter can remain available for a later external evaluation.

The notebook will stop instead of silently falling back to synthetic data.


In [ ]:
# 3) Real-data controls for a T4
DATA_PATH = "/content/fraudTrain.csv"
TARGET = "is_fraud"
ROWS = 100_000
SEEDS = [42, 123, 456]
EPOCHS = 10
PATIENCE = 4
BATCH_SIZE = 256

print({"DATA_PATH": DATA_PATH, "ROWS": ROWS, "SEEDS": SEEDS, "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE})


In [ ]:
# 4) Load REAL data through AutoData's ingestion path
from pathlib import Path
from src.ingestion.loader import load_dataset
from src.ingestion.roles import detect_roles, schema_for_profiling
from src.ingestion.schema_detector import detect_schema
from src.utils.config import ROOT, load_config

cfg = load_config()
path = Path(DATA_PATH)
if not path.exists():
    from google.colab import files
    print(f"{DATA_PATH} not found. Upload fraudTrain.csv (recommended).")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No dataset uploaded")
    path = Path('/content') / next(iter(uploaded))

if 'fraudtest' in path.name.lower():
    print("WARNING: You are using fraudTest.csv for development. Prefer fraudTrain.csv and reserve fraudTest.csv for external evaluation.")

ds = load_dataset(path, metadata_dir=ROOT / cfg["project"]["data_dir"] / "raw" / "_metadata")
df, dataset_id = ds.df, ds.metadata.dataset_id
schema = detect_schema(df, dataset_id)
roles = detect_roles(df, schema, target=TARGET)
ps = schema_for_profiling(schema, roles, df)

print(f"dataset={path.name} id={dataset_id}")
print(f"Rows={len(df):,}, columns={len(df.columns)}")
print(f"target={roles.target!r}, entity={roles.entity!r}, datetime={roles.datetime!r}")
assert roles.task == "binary_classification"


In [ ]:
# 5) Natural-prevalence validation/test; enriched train only
import copy, json
from src.preprocessing.levels import DataPreparer, infer_roles

level_roles = infer_roles(df, ps, roles)
validation_cfg = copy.deepcopy(cfg)
validation_cfg["sampling"]["train_positive_share"] = 0.10
validation_cfg["sampling"]["validation_positive_share"] = None
validation_cfg["sampling"]["keep_all_test_positives"] = False

preparer = DataPreparer(df, ps, level_roles, validation_cfg, rows=ROWS, seed=cfg["project"]["seed"])
split_info = preparer.prepare_split()
print(json.dumps(split_info["sample"], indent=2, default=str))

for split in ["validation", "test"]:
    r = split_info["sample"][split]
    print(split, "sample prevalence=", r["sample_positive_share"], "weighted prevalence=", r["weighted_positive_rate"])


In [ ]:
# 6) Build E2 once and REQUIRE the runtime point-in-time audit to pass
probe = preparer.build("E2")
pit = probe.info.get("point_in_time", {})
print(json.dumps(pit, indent=2, default=str))
assert pit.get("passed"), f"Point-in-time audit failed: {pit}"
for required in ["card_history", "merchant_history", "previous_transactions"]:
    if required in pit:
        assert pit[required]["passed"], f"{required} failed: {pit[required]}"
        assert pit[required].get("same_timestamp_policy") == "strictly_earlier_only"
print("PASS: enabled behavioral features use strictly earlier timestamps only.")


## Focused validation matrix

We deliberately avoid the full exploratory matrix. The six variants below answer the remaining questions:
- E0 raw baseline;
- E1 continuous (the representation that repeatedly beat E1 quantile);
- E2 full current baseline;
- E2 history only;
- E2 sequence only;
- E2 history + sequence (best 30k real-data variant).

All variants use the same split/sample seed and the same three model seeds.


In [ ]:
# 7) Focused GPU matrix
import copy, pandas as pd
from src.evaluation.experiment import run_comparison_multiseed, aggregate_seeds

settings = {"epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE}


def make_preparer(config):
    p = DataPreparer(df, ps, level_roles, config, rows=ROWS, seed=cfg["project"]["seed"])
    p.prepare_split()
    return p


def run_variant(name, level, config):
    print(f"\n=== {name} ===")
    p = make_preparer(config)
    if level == "E2":
        built = p.build("E2")
        pit = built.info.get("point_in_time", {})
        assert pit.get("passed"), f"PIT audit failed in {name}: {pit}"
    out = run_comparison_multiseed(
        p, [level], config, SEEDS, settings=settings, device="cuda",
        progress=lambda done,total,seed,r: print(
            f"  seed={seed}: weighted PR-AUC={r.metrics['test']['pr_auc']:.6f} | "
            f"unweighted PR-AUC={r.metrics['test_unweighted']['pr_auc']:.6f}")
    )
    table = aggregate_seeds(out).reset_index().rename(columns={"Level":"level"})
    table.insert(0, "variant", name)

    # Natural test samples have weights ~1, but save the explicitly unweighted metrics too.
    rows=[]
    for seed, results in out.items():
        r=results[0]
        m=r.metrics["test_unweighted"]
        rows.append({"seed":seed, **{k:m[k] for k in ["pr_auc","roc_auc","precision","recall","f1"]}})
    u=pd.DataFrame(rows)
    for metric in ["pr_auc","roc_auc","precision","recall","f1"]:
        table[f"natural_{metric} mean"] = u[metric].mean()
        table[f"natural_{metric} std"] = u[metric].std(ddof=1)
    return table

variants=[]

# E0 raw: current quantile representation
c=copy.deepcopy(validation_cfg)
variants.append(run_variant("E0_raw", "E0", c))

# E1: continuous numeric representation
c=copy.deepcopy(validation_cfg)
c["representation"]["numeric_mode"]="continuous"
c["representation"]["numeric_coarse_bins"]=0
variants.append(run_variant("E1_continuous", "E1", c))

# E2 full: current conservative quantile representation
c=copy.deepcopy(validation_cfg)
c["features"]["enabled_groups"]=["temporal","history","sequence"]
variants.append(run_variant("E2_full", "E2", c))

for groups, name in [
    (["history"], "E2_history"),
    (["sequence"], "E2_sequence"),
    (["history","sequence"], "E2_history_sequence"),
]:
    c=copy.deepcopy(validation_cfg)
    c["features"]["enabled_groups"]=groups
    variants.append(run_variant(name, "E2", c))

validation_table=pd.concat(variants, ignore_index=True)
display(validation_table.sort_values("natural_pr_auc mean", ascending=False))


In [ ]:
# 8) Stability + practical decision view
view = validation_table[[
    "variant", "level", "Seeds",
    "natural_pr_auc mean", "natural_pr_auc std",
    "natural_precision mean", "natural_recall mean", "natural_f1 mean",
    "pr_auc mean", "pr_auc std"
]].sort_values("natural_pr_auc mean", ascending=False)
display(view)

print("Decision rule: do not promote a configuration unless the improvement survives all 3 seeds and natural-prevalence evaluation.")


In [ ]:
# 9) Export results + audit context
from datetime import datetime
from pathlib import Path
import json, torch

out_dir = PROJECT_ROOT / "experiments" / "colab_gpu"
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = out_dir / f"validation_100k_{stamp}.csv"
json_path = out_dir / f"validation_100k_context_{stamp}.json"
validation_table.to_csv(csv_path, index=False)

context = {
    "dataset_id": dataset_id,
    "dataset_file": path.name,
    "rows_requested": ROWS,
    "seeds": SEEDS,
    "device": torch.cuda.get_device_name(0),
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "batch_size": BATCH_SIZE,
    "evaluation_distribution": "natural validation/test; fraud-enriched training only",
    "split": split_info,
    "point_in_time_audit": pit,
    "same_timestamp_policy": "strictly earlier only",
    "variants": validation_table["variant"].tolist(),
}
json_path.write_text(json.dumps(context, indent=2, default=str))
print(csv_path)
print(json_path)

from google.colab import files
files.download(str(csv_path))
files.download(str(json_path))


## After this run

Upload the two exported files back to ChatGPT. The next decision is evidence-based:
- if `E2_history_sequence` remains strongest and stable at natural prevalence, make it the risk-profile default;
- if `E2_full` is statistically indistinguishable, prefer the simpler/lower-latency option only after measuring feature count and inference cost;
- if the gain collapses, inspect sampling, cold-start slices, and individual history features before adding model complexity.

A later final evaluation should reserve an untouched source/time period (ideally `fraudTest.csv`) and avoid tuning on it.
